In [4]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint

def bioprocess_model(y, t, params):
    #Defining output of the model: X for biomass, G for glucose, L for lactate, N for Ammonia, P for product 
    X, G, L, N, P = y
   	#Parameters definition mu_max is the maximum speed of growth, Ks is the reaction rate constan of substrate consumption, Ki is the reaction rate constant for inhibition, qp production rate, mass of product per biomass, Yxl conversion rate biomass per substrate 
    mu_max, Ks, Ki, qp, Yxl = params
    
    mu = mu_max * (G / (Ks + G)) * (Ki / (Ki + L))
    
    dXdt = mu * X
    dGdt = -(mu / Yxl) * X
    dLdt = 1.8 * (mu / Yxl) * X 
    dNdt = 0.5 * mu * X
    dPdt = qp * X if t > 72 else 0
    
    return [dXdt, dGdt, dLdt, dNdt, dPdt]

def generate_industrial_dataset(n_batches=30):
    all_telemetry = []
    all_outcomes = []
    
	# creating a list from 0 to 240 hours with 240 timepoints in between, which is one point each hour
    t_eval = np.linspace(0, 240, 240) 
    
    for b_id in range(n_batches):
        	
	#creating a string batch ID based on the batch position within n_batches list, 03d is a formatter that ensures the number is always 3 digits long by padding it with zeros 
        batch_id = f"LOT_{b_id:03d}"
        
	# creating variation in biomass max growth rate and product production rate to mimic common biologic variability
        mu_max = np.random.normal(0.04, 0.002)
       	qp = np.random.normal(0.001, 0.0001)
        
    #Initial value for each integration is defined as a list. Biomass starts at 0.5 g/L, substrate at 25 g/L and other variable are at zero when starting.
        y0 = [0.5, 25.0, 0.0, 0.0, 0.0]
        	
	#Parameters are defined as tuples (not possible to be changed)
        params = (mu_max, 0.5, 40.0, qp, 0.4)
      
    #System of differential equations is fully defined here with model, initial conditions, time intervals and equation parameters  
        sol = odeint(bioprocess_model, y0, t_eval, args=(params,))

    #Building batch data frame by populating the equation system results, adding time values and batch ids        
        batch_df = pd.DataFrame(sol, columns=['VCD', 'Glucose', 'Lactate', 'Ammonia', 'Product'])
        batch_df['Hour'] = t_eval
        batch_df['Batch_ID'] = batch_id
        
    # Adding sensor noise and pH correlation
        batch_df['pH'] = 7.1 - (batch_df['Lactate'] * 0.02) + np.random.normal(0, 0.01, 240)
        batch_df['VCD'] = batch_df['VCD'] + np.random.normal(0, 0.1, 240)
        
    # Whole time series data frame
        all_telemetry.append(batch_df)
        
    # Outcome based on transfection success at Hour 72
    # transfection efficacy is function of the ammonia concentration at transfection time. The te efficiency is a list of the transfection efficiency for each batch
        te_efficiency = 1.0 - (batch_df.iloc[72]['Ammonia'] / 20.0)
        	
    #Main outcomes only data frame
        all_outcomes.append({
            'Batch_ID': batch_id,
           #taking the last entry of production concentration and using transfection efficiency for actual product concentration
             'Final_Titer': batch_df['Product'].iloc[-1] * te_efficiency,
            #taking the maximum between 0.05 and calculated ratio
            'Full_Empty_Ratio': max(0.05, 0.3 * te_efficiency)
        })
     
    return pd.concat(all_telemetry), pd.DataFrame(all_outcomes)

#because the generate industrial dataset has a default parameter of 30 batches, this line produces both data frame for 30 batches.
df_telemetry, df_outcomes = generate_industrial_dataset()


In [12]:
df_telemetry.to_csv('telemetry_batches.csv', index=False)
df_outcomes.to_csv('batch_outcomes.csv', index=False)